# IT Support Dashboard - Supporting Notebook 1

This notebook is the project entry point for data quality assurance. Within the wider IT support ticket analysis workflow, it preserves a traceable record ID, removes unusable text records, corrects malformed tags, standardises key fields, and produces the clean dataset used by the downstream representativeness, lemmatisation, clustering, and reporting notebooks.

## Cleaning Process

Notebook 1 documents the project-wide cleaning and validation layer for the raw ticket export. The process:

- Preserves the original source row position as an explicit record identifier for auditability across outputs
- Removes records missing either `answer` or `subject`, because both fields are required downstream
- Corrects 13 records containing comma-separated tags
- Identifies and corrects tags containing missing word boundaries
- Standardises `priority`, `language`, and long-text fields
- Optimises data types across analytical dimensions
- Produces clean CSV and Parquet outputs for downstream notebooks
- Saves audit tables showing which records were removed or amended

### Importing

The required packages are imported, the project directories are verified through the configuration file, and the raw dataset is loaded.

In [3]:
# Importing packages
import os
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Add the project root to sys.path by searching upward for config.py
project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "config.py").exists()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing config.py.")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import Config  # noqa: E402

Config.ensure_directories()

# Load the raw data and retain its original row position for traceability
df = pd.read_csv(
    Config.RAW_DATA_PATH,
    encoding="utf-8",
    na_values=[
        "",
        " ",
        "NA",
        "N/A",
        "na",
        "n/a",
        "NULL",
        "null",
        "None",
        "none",
        "NAN",
        "NaN",
        "nan",
    ],
).reset_index(names="index")

print(f"\nImported dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())

[Config] Verified project directory structure under C:\Users\cosmi\Documents\Projects\IT-Support-Ticket-Analysis

Imported dataset: 28,587 rows × 17 columns


,index,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


### Assessing data quality

The raw dataset was assessed by reviewing its shape, missing values, data types, and memory usage.

It initially contained 28,587 rows and 17 columns. There were 7 null values in `answer`, 3,838 in `subject`, and 13 in `tag_2`. Missing tags were retained because tag fields are optional. Records missing either `subject` or `answer` were excluded because both fields are required by the downstream text analysis.

In [4]:
print("Dataset shape:")
print(df.shape)
print("\nMissing values by column:")
print(df.isna().sum().sort_values())
print("\nData types by column:")
print(df.dtypes)
print("\nMemory usage by column (bytes):")
print(df.memory_usage(deep=True))

Dataset shape:
(28587, 17)

Missing values by column:
index           0
body            0
queue           0
type            0
priority        0
language        0
tag_1           0
version         0
answer          7
tag_2          13
tag_3         136
tag_4        3058
subject      3838
tag_5       14042
tag_6       22713
tag_7       26547
tag_8       28022
dtype: int64

Data types by column:
index       int64
subject       str
body          str
answer        str
type          str
queue         str
priority      str
language      str
version     int64
tag_1         str
tag_2         str
tag_3         str
tag_4         str
tag_5         str
tag_6         str
tag_7         str
tag_8         str
dtype: object

Memory usage by column (bytes):
Index            132
index         228696
subject      1338793
body        11367754
answer      11361229
type          437349
queue         705930
priority      360180
language      285870
version       228696
tag_1         432901
tag_2         461790

### Removing records with missing required text

The downstream analyses require both a ticket subject and an agent response. The raw data contained:

- 7 records with a missing `answer`
- 3,838 records with a missing `subject`
- 1 record (source row 13,651) missing both fields

Removing records missing either field excluded 3,844 rows, or 13.45% of the raw dataset. The explicit `index` field was retained so each excluded record could be traced to its position in the source export.

In [5]:
# Identify records missing either required text field
missing_required_text_mask = df["answer"].isna() | df["subject"].isna()
missing_required_text_df = (
    df.loc[missing_required_text_mask]
    .copy()
    .sort_values("index")
)

missing_required_text_df.to_csv(
    Config.MISSING_REQUIRED_TEXT_PATH,
    index=False,
    encoding="latin-1",
)

missing_required_text_count = len(missing_required_text_df)
missing_required_text_percentage = 100 * missing_required_text_count / len(df)

print(
    f"\nRecords missing required text: {missing_required_text_count:,} "
    f"({missing_required_text_percentage:.2f}% of the raw dataset)"
)
display(missing_required_text_df[["index", "subject", "answer"]])


Records missing required text: 3,844 (13.45% of the raw dataset)


,index,subject,answer
870,870,NaN,Please provide detailed integration instructio...
887,887,NaN,"<name>, I regret to hear about the security br..."
888,888,NaN,Überprüfen Sie die Kampagnenberichte und verei...
915,915,NaN,"<name>, thank you for your email regarding the..."
929,929,NaN,We have received a report about a security inc...
...,...,...,...
28550,28550,NaN,We appreciate you bringing this critical issue...
28552,28552,NaN,"geehrter [name], wir danken Ihnen für Ihre Anf..."
28568,28568,NaN,"Wir bieten Social-Media-Management, Suchmaschi..."
28569,28569,NaN,Wir haben Ihre E-Mail über den Leistungsriss b...


In [6]:
# Remove records missing Answer or Subject while retaining source row identifiers
dropped_nulls_df = df.dropna(subset=["answer", "subject"]).copy()
rows_removed = len(df) - len(dropped_nulls_df)

if rows_removed != missing_required_text_count:
    raise ValueError("The number of removed records does not match the audit table.")

if dropped_nulls_df[["answer", "subject"]].isna().any().any():
    raise ValueError("Required text fields still contain null values after filtering.")

print(
    f"\nRows before removal: {len(df):,}\n"
    f"Rows after removal:  {len(dropped_nulls_df):,}\n"
    f"Rows removed:        {rows_removed:,}"
)


Rows before removal: 28,587
Rows after removal:  24,743
Rows removed:        3,844


### Correcting comma-separated tags

Tag fields should contain one value per column. In 12 records, `tag_1` contained multiple comma-separated values; one further record had the same issue in `tag_3`. The affected values were split and redistributed across the available tag columns while preserving their original order. No other tag columns contained comma-separated values.

In [7]:
# Identify tag columns and sort them by their numeric suffix.
tag_columns = sorted(
    (
        column
        for column in dropped_nulls_df.columns
        if column.startswith("tag_")
    ),
    key=lambda column: (
        int(column.removeprefix("tag_"))
        if column.removeprefix("tag_").isdigit()
        else float("inf")
    ),
)

print(f"\nNumber of tag columns: {len(tag_columns)}")


# Identify malformed tag cells
multiple_tag_mask = (
    dropped_nulls_df[tag_columns]
    .astype("string")
    .apply(lambda column: column.str.contains(",", regex=False, na=False))
)

# Store affected dataframe row indices for each tag column.
long_tags_dict = {
    tag_column: multiple_tag_mask.index[multiple_tag_mask[tag_column]].tolist()
    for tag_column in tag_columns
    if multiple_tag_mask[tag_column].any()
}

# Unique dataframe indices containing at least one malformed tag cell.
affected_row_indices = multiple_tag_mask.any(axis=1)
affected_row_indices = multiple_tag_mask.index[affected_row_indices].tolist()


# Report findings
for tag_column in tag_columns:
    affected_indices = long_tags_dict.get(tag_column, [])

    if not affected_indices:
        print(f"\nNo rows with multiple tags found in {tag_column}.")
        continue

    print(
        f"\nTable containing {len(affected_indices)} row(s) "
        f"with multiple tags within {tag_column}:"
    )

    display(
        dropped_nulls_df.loc[
            affected_indices,
            ["index", *tag_columns],
        ].sort_values("index")
    )


print(
    f"\nColumns {list(long_tags_dict)} contain multiple tags, "
    "separated by commas."
)

print(
    f"Total unique affected records: {len(affected_row_indices)}"
)


# Save affected records for audit
invalid_tags_df = (dropped_nulls_df.loc[affected_row_indices].sort_values("index").copy())
invalid_tags_df.to_csv(Config.INVALID_TAGS_PATH, index=False, encoding="latin-1")


Number of tag columns: 8

Table containing 12 row(s) with multiple tags within tag_1:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
958,958,"Performance,Bug,Disruption,Security",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1784,1784,"Performance,Disruption,Outage,Monitoring,Analysis",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2166,2166,"Crash,Performance,Outage,Disruption,Recovery,S...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2331,2331,"Performance,Disruption,IT,Tech Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2410,2410,"Performance,Disruption,Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3146,3146,"Performance,Outage,Disruption,Recovery,Marketi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3537,3537,"Security,IT,Tech Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4758,4758,"Performance,Disruption,Outage,Support,Integration",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5355,5355,"Performance,Security,Feature,Documentation",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5698,5698,"Security,IT,Tech Support,Data Privacy,Regulati...",NaN,NaN,NaN,NaN,NaN,NaN,NaN



No rows with multiple tags found in tag_2.

Table containing 1 row(s) with multiple tags within tag_3:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
5885,5885,Security,Network,"Disruption,IT",Tech Support,NaN,NaN,NaN,NaN



No rows with multiple tags found in tag_4.

No rows with multiple tags found in tag_5.

No rows with multiple tags found in tag_6.

No rows with multiple tags found in tag_7.

No rows with multiple tags found in tag_8.

Columns ['tag_1', 'tag_3'] contain multiple tags, separated by commas.
Total unique affected records: 13


In [8]:
# Create a copy before correcting comma-separated tags
fixed_tags_df = dropped_nulls_df.copy()

max_tag_columns = len(tag_columns)
overflow_rows = []

for row_index in affected_row_indices:
    corrected_tags = []

    for tag_column in tag_columns:
        value = fixed_tags_df.at[row_index, tag_column]

        if pd.isna(value):
            continue

        corrected_tags.extend(
            tag.strip()
            for tag in str(value).split(",")
            if tag.strip()
        )

    if len(corrected_tags) > max_tag_columns:
        overflow_rows.append(
            {
                "index": fixed_tags_df.at[row_index, "index"],
                "overflow_tags": corrected_tags[max_tag_columns:],
            }
        )
        continue

    corrected_tags.extend([pd.NA] * (max_tag_columns - len(corrected_tags)))
    fixed_tags_df.loc[row_index, tag_columns] = corrected_tags

if overflow_rows:
    overflow_df = pd.DataFrame(overflow_rows)
    display(overflow_df)
    raise ValueError(
        f"{len(overflow_rows)} record(s) contain more tags than the "
        f"{max_tag_columns} available tag columns."
    )

remaining_comma_mask = (
    fixed_tags_df.loc[affected_row_indices, tag_columns]
    .astype("string")
    .apply(lambda column: column.str.contains(",", regex=False, na=False))
)

if remaining_comma_mask.any().any():
    raise ValueError("Comma-separated tags remain after correction.")

comma_corrections_df = (
    fixed_tags_df.loc[affected_row_indices]
    .sort_values("index")
    .copy()
)

comma_corrections_df.to_csv(
    Config.FIXED_COMMA_TAGS_PATH,
    index=False,
    encoding="latin-1",
)

print(
    f"\nSuccessfully corrected comma-separated tags in "
    f"{len(affected_row_indices):,} record(s)."
)
display(comma_corrections_df[["index", *tag_columns]])


Successfully corrected comma-separated tags in 13 record(s).


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
958,958,Performance,Bug,Disruption,Security,NaN,NaN,NaN,NaN
1784,1784,Performance,Disruption,Outage,Monitoring,Analysis,NaN,NaN,NaN
2166,2166,Crash,Performance,Outage,Disruption,Recovery,Server,DataProcessing,NaN
2331,2331,Performance,Disruption,IT,Tech Support,NaN,NaN,NaN,NaN
2410,2410,Performance,Disruption,Support,NaN,NaN,NaN,NaN,NaN
3146,3146,Performance,Outage,Disruption,Recovery,Marketing,Agentur,Analyse,NaN
3537,3537,Security,IT,Tech Support,NaN,NaN,NaN,NaN,NaN
4758,4758,Performance,Disruption,Outage,Support,Integration,NaN,NaN,NaN
5355,5355,Performance,Security,Feature,Documentation,NaN,NaN,NaN,NaN
5698,5698,Security,IT,Tech Support,Data Privacy,Regulation,Patient Data,Threat Prevention,NaN


### Correcting missing spaces in tags

Tags containing suspected missing word boundaries were then identified. For example, `DataProcessing` was treated as malformed and corrected to `Data Processing`. Known mixed-case technology names such as `SaaS`, `IoT`, and `PostgreSQL` were excluded from this rule.

In [9]:
# Identify tags containing a lowercase-to-uppercase transition
missing_space_pattern = r"[a-z][A-Z]"
missing_space_boundary = re.compile(r"(?<=[a-z])(?=[A-Z])")

# Known valid mixed-case tags that should not be flagged
valid_mixed_case_tags = {
    "SaaS",
    "SaaS Platform",
    "Cloud SaaS",
    "IoT",
    "macOS",
    "PostgreSQL",
    "PyTorch",
    "MySQL",
}

grammar_tags_dict = {}

for tag_column in tag_columns:
    tag_series = fixed_tags_df[tag_column].astype("string")
    missing_space_mask = (
        tag_series.str.contains(missing_space_pattern, regex=True, na=False)
        & ~tag_series.isin(valid_mixed_case_tags)
    )

    affected_indices = fixed_tags_df.index[missing_space_mask].tolist()

    if not affected_indices:
        print(f"\nNo suspected missing spaces found in {tag_column}.")
        continue

    grammar_tags_dict[tag_column] = affected_indices
    affected_rows = (
        fixed_tags_df.loc[affected_indices, ["index", *tag_columns]]
        .sort_values("index")
    )

    print(
        f"\nFound {len(affected_indices):,} record(s) with suspected "
        f"missing spaces in {tag_column}:"
    )
    display(affected_rows)

grammar_affected_indices = sorted(
    {
        row_index
        for affected_indices in grammar_tags_dict.values()
        for row_index in affected_indices
    }
)

grammar_tags_df = (
    fixed_tags_df.loc[grammar_affected_indices]
    .sort_values("index")
    .copy()
)
grammar_tags_df.to_csv(
    Config.MALFORMED_TAGS_PATH,
    index=False,
    encoding="latin-1",
)

print(f"\nTag columns requiring review: {list(grammar_tags_dict)}")
print(f"Total distinct affected records: {len(grammar_affected_indices):,}")


No suspected missing spaces found in tag_1.

Found 5 record(s) with suspected missing spaces in tag_2:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
3621,3621,Support,ActiveCampaign,Strategy,Audience,NaN,NaN,NaN,NaN
3856,3856,Security,DataProtection,Confidentiality,Integration,Encryption,Compliance,MedicalData,NaN
5593,5593,Security,DataProtection,Healthcare,Encryption,AccessManagement,NaN,NaN,NaN
9086,9086,Security,DataLeak,Investigation,SystemVulnerability,Privacy,PatientData,SecurityAudit,NaN
9742,9742,Security,DataProtection,Confidentiality,Healthcare,Compliance,SystemSecurity,PatientPrivacy,NaN



Found 2 record(s) with suspected missing spaces in tag_3:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
3710,3710,Security,IT,DataProtection,Cybersecurity,HospitalInfrastructure,SensitiveData,ThreatPrevention,NaN
7561,7561,Security,Malware,HealthIT,Threat,Incident,NaN,NaN,NaN



Found 14 record(s) with suspected missing spaces in tag_4:


...


Found 5 record(s) with suspected missing spaces in tag_7:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
2166,2166,Crash,Performance,Outage,Disruption,Recovery,Server,DataProcessing,NaN
3710,3710,Security,IT,DataProtection,Cybersecurity,HospitalInfrastructure,SensitiveData,ThreatPrevention,NaN
3856,3856,Security,DataProtection,Confidentiality,Integration,Encryption,Compliance,MedicalData,NaN
9086,9086,Security,DataLeak,Investigation,SystemVulnerability,Privacy,PatientData,SecurityAudit,NaN
9742,9742,Security,DataProtection,Confidentiality,Healthcare,Compliance,SystemSecurity,PatientPrivacy,NaN



Found 2 record(s) with suspected missing spaces in tag_8:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
5803,5803,Bug,Performance,Hardware,Support,Update,Restart,Cache,SystemSlowdown
7732,7732,Security,Patient,Privacy,DataProtection,Healthcare,Confidentiality,Safety,MedicalInformation



Tag columns requiring review: ['tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']
Total distinct affected records: 30


In [10]:
# Correct missing spaces in the identified tag values
spacing_corrections = []

for tag_column, affected_indices in grammar_tags_dict.items():
    for row_index in affected_indices:
        original_value = fixed_tags_df.at[row_index, tag_column]
        corrected_value = missing_space_boundary.sub(" ", str(original_value))

        fixed_tags_df.at[row_index, tag_column] = corrected_value
        spacing_corrections.append(
            {
                "index": fixed_tags_df.at[row_index, "index"],
                "tag_column": tag_column,
                "original_value": original_value,
                "corrected_value": corrected_value,
            }
        )

spacing_corrections_df = pd.DataFrame(
    spacing_corrections,
    columns=["index", "tag_column", "original_value", "corrected_value"],
).sort_values(["index", "tag_column"]).reset_index(drop=True)

spacing_corrections_df.to_csv(
    Config.FIXED_TAGS_PATH,
    index=False,
    encoding="latin-1",
)

print(
    f"\nCorrected {len(spacing_corrections_df):,} tag value(s) "
    f"across {len(grammar_tags_dict):,} column(s)."
)
display(spacing_corrections_df)


Corrected 47 tag value(s) across 7 column(s).


,index,tag_column,original_value,corrected_value
0,2166,tag_7,DataProcessing,Data Processing
1,3621,tag_2,ActiveCampaign,Active Campaign
2,3710,tag_3,DataProtection,Data Protection
3,3710,tag_5,HospitalInfrastructure,Hospital Infrastructure
4,3710,tag_6,SensitiveData,Sensitive Data
5,3710,tag_7,ThreatPrevention,Threat Prevention
6,3856,tag_2,DataProtection,Data Protection
7,3856,tag_7,MedicalData,Medical Data
8,3978,tag_4,SoftwareProblem,Software Problem
9,3996,tag_4,DataProcessing,Data Processing


The corrected records were rechecked to confirm that no identified missing-space patterns remained.

In [11]:
# Confirm that no identified values still contain missing-space patterns
remaining_spacing_issues = []

for tag_column, affected_indices in grammar_tags_dict.items():
    corrected_tag_series = fixed_tags_df.loc[affected_indices, tag_column].astype("string")
    unresolved_mask = (
        corrected_tag_series.str.contains(missing_space_pattern, regex=True, na=False)
        & ~corrected_tag_series.isin(valid_mixed_case_tags)
    )

    for row_index in unresolved_mask.index[unresolved_mask]:
        remaining_spacing_issues.append(
            {
                "index": fixed_tags_df.at[row_index, "index"],
                "tag_column": tag_column,
                "value": fixed_tags_df.at[row_index, tag_column],
            }
        )

if grammar_affected_indices:
    corrected_spacing_records = (
        fixed_tags_df.loc[grammar_affected_indices, ["index", *tag_columns]]
        .sort_values("index")
    )
    print(
        f"\nRecords checked after correcting missing spaces "
        f"({len(corrected_spacing_records):,}):"
    )
    display(corrected_spacing_records)
else:
    print("\nNo records required missing-space corrections.")

if remaining_spacing_issues:
    remaining_spacing_issues_df = (
        pd.DataFrame(remaining_spacing_issues)
        .sort_values(["index", "tag_column"])
        .reset_index(drop=True)
    )
    print("\nUnresolved missing-space tag issues:")
    display(remaining_spacing_issues_df)
    raise ValueError(
        f"{len(remaining_spacing_issues_df):,} missing-space tag "
        "issue(s) remain after correction."
    )

all_corrected_tag_indices = sorted(
    set(affected_row_indices) | set(grammar_affected_indices)
)
validated_tags_df = (
    fixed_tags_df.loc[all_corrected_tag_indices]
    .sort_values("index")
    .copy()
)
validated_tags_df.to_csv(
    Config.VALIDATED_TAGS_PATH,
    index=False,
    encoding="latin-1",
)

print(
    f"\nSaved {len(validated_tags_df):,} fully validated tag record(s) "
    f"to {Config.VALIDATED_TAGS_PATH.name}."
)

print(
    f"\nAll missing-space corrections were validated successfully "
    f"across {len(grammar_affected_indices):,} record(s)."
)


Records checked after correcting missing spaces (30):


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
2166,2166,Crash,Performance,Outage,Disruption,Recovery,Server,Data Processing,NaN
3621,3621,Support,Active Campaign,Strategy,Audience,NaN,NaN,NaN,NaN
3710,3710,Security,IT,Data Protection,Cybersecurity,Hospital Infrastructure,Sensitive Data,Threat Prevention,NaN
3856,3856,Security,Data Protection,Confidentiality,Integration,Encryption,Compliance,Medical Data,NaN
3978,3978,Performance,Network,Bug,Software Problem,Disruption,NaN,NaN,NaN
3996,3996,Performance,Disruption,Recovery,Data Processing,NaN,NaN,NaN,NaN
4925,4925,Outage,Disruption,Network,Security,Software Conflict,NaN,NaN,NaN
5593,5593,Security,Data Protection,Healthcare,Encryption,Access Management,NaN,NaN,NaN
5687,5687,Product,Documentation,Support,Integration,Data Robot,NaN,NaN,NaN
5803,5803,Bug,Performance,Hardware,Support,Update,Restart,Cache,System Slowdown



Saved 42 fully validated tag record(s) to 06_Validated_Tags.csv.

All missing-space corrections were validated successfully across 30 record(s).


### Standardising inputs

The dataset was standardised by normalising `priority` and `language`, trimming surrounding whitespace from `subject`, removing formatting artefacts from `answer` and `body`, and optimising data types for downstream analysis. The explicit `index` field retains each record's position in the source export.

In [12]:
# Create the standardised dataset
standardised_text_df = fixed_tags_df.copy()

required_columns = {
    "index",
    "subject",
    "body",
    "answer",
    "type",
    "queue",
    "priority",
    "language",
    "version",
    *tag_columns,
}
missing_columns = required_columns.difference(standardised_text_df.columns)

if missing_columns:
    raise KeyError(f"Required columns are missing: {sorted(missing_columns)}")

standardised_text_df["priority"] = (
    standardised_text_df["priority"].astype("string").str.strip().str.title()
)
standardised_text_df["language"] = (
    standardised_text_df["language"].astype("string").str.strip().str.upper()
)
standardised_text_df["subject"] = (
    standardised_text_df["subject"].astype("string").str.strip()
)

for column in ["body", "answer"]:
    standardised_text_df[column] = (
        standardised_text_df[column]
        .astype("string")
        .str.replace(r"(?i)<br\s*/?>", " ", regex=True)
        .str.replace(r"\\r\\n|\\n|\\r|\r\n|\n|\r", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

standardised_text_df = standardised_text_df.rename(columns=str.capitalize)

category_columns = [
    "Type",
    "Queue",
    "Priority",
    "Language",
    "Version",
    *[column.capitalize() for column in tag_columns],
]
string_columns = ["Subject", "Body", "Answer"]
dtype_map = {
    **{column: "category" for column in category_columns},
    **{column: "string[pyarrow]" for column in string_columns},
}

standardised_text_df = standardised_text_df.astype(dtype_map)

print(
    f"\nStandardised dataset: {standardised_text_df.shape[0]:,} rows × "
    f"{standardised_text_df.shape[1]:,} columns"
)
display(standardised_text_df.head())


Standardised dataset: 24,743 rows × 17 columns


,Index,Subject,Body,Answer,Type,Queue,Priority,Language,Version,Tag_1,Tag_2,Tag_3,Tag_4,Tag_5,Tag_6,Tag_7,Tag_8
0,0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team, ich möchte einen g...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,High,DE,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,1,Account Disruption,"Dear Customer Support Team, I am writing to re...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,High,EN,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team, I hope this messag...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,Medium,EN,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,3,Inquiry Regarding Invoice Details,"Dear Customer Support Team, I hope this messag...",We appreciate you reaching out with your billi...,Request,Billing and Payments,Low,EN,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,4,Question About Marketing Agency Software Compa...,"Dear Support Team, I hope this message reaches...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,Medium,EN,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


### Validation

The standardised dataset was revalidated before export. These checks enforce record-count and identifier integrity, required-text completeness, expected categorical values, and the absence of comma-separated tag values.

In [13]:
expected_rows = len(df) - missing_required_text_count
expected_priorities = {"Low", "Medium", "High"}
expected_languages = {"EN", "DE"}
exported_tag_columns = [column.capitalize() for column in tag_columns]

if len(standardised_text_df) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows:,} clean records, found "
        f"{len(standardised_text_df):,}."
    )

if standardised_text_df["Index"].isna().any():
    raise ValueError("Source record identifiers contain null values.")

if standardised_text_df["Index"].duplicated().any():
    raise ValueError("Duplicate source record identifiers were found.")

if standardised_text_df[["Subject", "Answer"]].isna().any().any():
    raise ValueError("Subject or Answer still contains null values.")

if not set(standardised_text_df["Priority"].dropna()).issubset(expected_priorities):
    raise ValueError("Unexpected Priority values were found.")

if not set(standardised_text_df["Language"].dropna()).issubset(expected_languages):
    raise ValueError("Unexpected Language values were found.")

remaining_commas = (
    standardised_text_df[exported_tag_columns]
    .astype("string")
    .apply(lambda column: column.str.contains(",", regex=False, na=False))
)

if remaining_commas.any().any():
    raise ValueError("Comma-separated tag values remain in the dataset.")

print("All final data-quality checks passed.")
print(f"\nDataset shape: {standardised_text_df.shape}")
print("\nMissing values by column:")
print(standardised_text_df.isna().sum().sort_values())
print("\nData types by column:")
print(standardised_text_df.dtypes)
print("\nMemory usage by column (bytes):")
print(standardised_text_df.memory_usage(deep=True))

All final data-quality checks passed.

Dataset shape: (24743, 17)

Missing values by column:
Index           0
Subject         0
Body            0
Answer          0
Type            0
Queue           0
Priority        0
Language        0
Version         0
Tag_1           0
Tag_2           0
Tag_3          96
Tag_4        2629
Tag_5       12322
Tag_6       19879
Tag_7       23147
Tag_8       24333
dtype: int64

Data types by column:
Index          int64
Subject       string
Body          string
Answer        string
Type        category
Queue       category
Priority    category
Language    category
Version     category
Tag_1       category
Tag_2       category
Tag_3       category
Tag_4       category
Tag_5       category
Tag_6       category
Tag_7       category
Tag_8       category
dtype: object

Memory usage by column (bytes):
Index        726368
Index        197944
Subject     1306950
Body        9609289
Answer      9759134
Type          24804
Queue         25004
Priority      24781
L

The number of distinct values and the ten most frequent values in each column were reviewed as a final descriptive check.

In [14]:
# Provide a compact unique-value summary for each column
for column in standardised_text_df.columns:
    distinct_count = standardised_text_df[column].nunique(dropna=False)
    top_value_counts = standardised_text_df[column].value_counts(dropna=False).head(10)

    print(f"\n{column}: {distinct_count:,} unique values")
    print("Top 10 values (including nulls):")
    print(top_value_counts)


Index: 24,743 unique values
Top 10 values (including nulls):
Index
0    1
1    1
2    1
3    1
4    1
5    1
6    1
7    1
8    1
9    1
Name: count, dtype: int64

Subject: 24,742 unique values
Top 10 values (including nulls):
Subject
Assistance Request                                                        2
Wesentlicher Sicherheitsvorfall                                           1
Account Disruption                                                        1
Query About Smart Home System Integration Features                        1
Inquiry Regarding Invoice Details                                         1
Question About Marketing Agency Software Compatibility                    1
Feature Query                                                             1
System Interruptions                                                      1
Connectivity Problems with Printer on MacBook Pro                         1
Anfrage nach detaillierten Angaben zur Systemarchitektur der Plattform    1
Name

### Saving the clean dataset

The clean dataset is exported as CSV for broad compatibility and Parquet for efficient storage and downstream loading. The `Index` field retains the original source row position for traceability; the temporary pandas index is not exported.

In [15]:
# CSV – portable
standardised_text_df.to_csv(Config.CLEAN_CSV_PATH, index=False, encoding="latin-1")

# Parquet – efficient
standardised_text_df.to_parquet(
    Config.CLEAN_PARQUET_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

# Summary of saved files
for path in [Config.RAW_DATA_PATH, Config.CLEAN_CSV_PATH, Config.CLEAN_PARQUET_PATH]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{path.name:<25} | {size_mb:>6.2f} MB")

print("\nAll files saved successfully.")

IT_Tickets_Raw.csv        |  24.79 MB
07_Tickets_Clean.csv      |  21.32 MB
07_Tickets_Clean.parquet  |   9.85 MB

All files saved successfully.
